# Evaluate stella_en_1.5B_v5 on Colab T4

Companion to `docs/embedder-swap-runbook.md`, step 6. Encodes val-split queries on a T4 and hits your local Postgres via ngrok TCP to score `dense_only` and `hybrid_rrf` against the new 1024-dim embeddings.

**Before running:** select `Runtime → Change runtime type → T4 GPU`.

## 1. Fill in connection details

Paste the host/port from the `ngrok tcp 25432` window on your laptop (the port changes on every ngrok restart), and your Postgres credentials.

In [ ]:
NGROK_HOST = "0.tcp.eu.ngrok.io"   # <-- replace
NGROK_PORT = 29549                   # <-- replace
POSTGRES_USER = "mario"
POSTGRES_PASSWORD = "10diploma10"
POSTGRES_DB = "papers_db"

REPO_URL = "https://github.com/mariosam23/missing-citations-identifier"
BRANCH = "new_implementation"

import os
# TCP keepalives keep the ngrok tunnel pinned during the 5-min retrieval phase.
# Without these the main eval session sits idle and the rollback at __exit__ blows up.
_KEEPALIVES = "keepalives=1&keepalives_idle=30&keepalives_interval=10&keepalives_count=5"
os.environ["DB_URL"] = (
    f"postgresql+psycopg://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{NGROK_HOST}:{NGROK_PORT}/{POSTGRES_DB}?{_KEEPALIVES}"
)
os.environ["EMBEDDER_MODEL_NAME"] = "dunzhang/stella_en_1.5B_v5"
os.environ["EMBEDDER_DIM"] = "1024"
os.environ["EMBEDDER_BATCH_SIZE"] = "16"  # T4 safe for stella 1.5B at fp16
os.environ["EMBEDDER_DEVICE"] = "cuda"
print("DB target:", os.environ["DB_URL"].split('@')[1])

## 2. Clone repo and install dependencies

Same dependency set as the embed notebook. Colab ships Python 3.11; we skip `pip install -e .` so the 3.13 pin in `pyproject.toml` does not block us.

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!pip install -q --upgrade pip
!pip install -q     'sentence-transformers>=3.2' 'torch>=2.4'     'sqlalchemy>=2.0' 'psycopg[binary]>=3.2' 'pgvector>=0.3.6'     'pydantic>=2.7' 'pydantic-settings>=2.4'     'typer>=0.12' 'tqdm>=4.66' xformers einops
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 3. Upload the eval split

`data/eval/split_42.json` is **not committed**, so the clone does not include it. Upload it via the Files panel (folder icon on the left → upload to the session) into `/content/repo/data/eval/split_42.json`, then run the cell below to verify.

In [ ]:
from pathlib import Path
import json

split_path = Path("/content/repo/data/eval/split_42.json")
assert split_path.exists(), f"missing: {split_path} — upload it via the Files panel"
split = json.loads(split_path.read_text(encoding="utf-8"))
print("seed:", split.get("seed"))
print("val papers :", len(split.get("val_paper_ids", [])))
print("test papers:", len(split.get("test_paper_ids", [])))

## 4. Smoke test the tunnel + embedder

Confirms the ngrok tunnel is alive and stella loads on CUDA. If this hangs on the SQL call, ngrok is down or the credentials are wrong. If it OOMs, switch to a fresh runtime.

In [ ]:
import sys
sys.path.insert(0, '/content/repo/src')
import numpy as np
from sqlalchemy import text
from database.postgres.engine import get_session
from pipeline.embedding.embedder import encode_query, get_embedder

_ = get_embedder()
vec = encode_query("Self-attention powers the transformer architecture.")
print("query vec:", vec.shape, "L2:", float(np.linalg.norm(vec)))

with get_session() as session:
    n = session.execute(text("SELECT COUNT(*) FROM citation_context_embeddings")).scalar_one()
print("corpus context embeddings:", n)

# Free the model from this kernel so the eval subprocess has a clean GPU.
# Two copies of stella OOM a T4; this drops VRAM back near zero before cell 10.
import gc, torch
import pipeline.embedding.embedder as _emb
_emb._model = None
gc.collect()
torch.cuda.empty_cache()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 5. Run both eval variants on val

Writes JSON reports under `/content/repo/data/eval/reports/`. Each variant re-encodes every val query (~5–6 k contexts) on the T4 and runs retrieval against the laptop DB over ngrok.

If you see DB connection timeouts, drop `--workers` from 4 to 2 — the free ngrok tunnel can choke on too many concurrent sessions.

In [ ]:
!mkdir -p /content/repo/data/eval/reports
!cd /content/repo && PYTORCH_ALLOC_CONF=expandable_segments:True PYTHONPATH=src python -m scripts.evaluate     --variant dense_only --split val --workers 2     --output data/eval/reports/dense_only_stella_1p5b_val.json

In [ ]:
!cd /content/repo && PYTORCH_ALLOC_CONF=expandable_segments:True PYTHONPATH=src python -m scripts.evaluate     --variant hybrid_rrf --split val --workers 2     --output data/eval/reports/hybrid_rrf_stella_1p5b_val.json

## 6. Download the reports

Run the cell below — it triggers a browser download for each report. Save them into `data/eval/reports/` on the laptop so `scripts.diff_eval_reports` can find them.

In [ ]:
from google.colab import files
files.download("/content/repo/data/eval/reports/dense_only_stella_1p5b_val.json")
files.download("/content/repo/data/eval/reports/hybrid_rrf_stella_1p5b_val.json")